# 대피도 YOLO Round 3 — 핵심 클래스 보강

Round 2 `best.pt`에서 이어서 `exit / stair / you_are_here` 실제 데이터 부족을 보완합니다. **자동 라벨은 반드시 사람이 검수해야 합니다.**

In [ ]:
# 0. 환경 준비 - 자동검수 Round 3 수정본

!pip -q install ultralytics pillow numpy requests

from pathlib import Path
import shutil, zipfile, json, os, sys
from google.colab import files, drive
from ultralytics import YOLO

# 새 자동검수 키트 경로
KIT = Path("/content/evac_round3_auto_kit")

# 이미 압축이 풀려 있지 않을 때만 업로드
if not KIT.exists():
    print("자동검수 키트 ZIP을 업로드하세요.")
    uploaded = files.upload()

    zip_name = next(
        name for name in uploaded
        if name.lower().endswith(".zip")
    )

    print("업로드:", zip_name)

    with zipfile.ZipFile(zip_name, "r") as z:
        z.extractall("/content")

print("\nKIT =", KIT)
print("KIT 존재:", KIT.exists())

NAMES = [
    "exit",
    "stair",
    "elevator",
    "extinguisher",
    "hydrant",
    "you_are_here",
    "door",
    "room",
]

SEED = KIT / "seed_model" / "best_round2.pt"

IMAGE_SIZE = 960
BATCH = -1

print("수집 스크립트:", (KIT / "scripts" / "collect_round3_real.py").exists())
print("자동검수 스크립트:", (KIT / "scripts" / "auto_pseudo_review.py").exists())
print("Round2 모델:", SEED.exists())

assert KIT.exists(), f"KIT 폴더가 없습니다: {KIT}"
assert (KIT / "scripts" / "collect_round3_real.py").exists()
assert (KIT / "scripts" / "auto_pseudo_review.py").exists()
assert SEED.exists(), f"모델이 없습니다: {SEED}"

print("\n✅ 0번 환경 준비 완료")

In [ ]:
from pathlib import Path

KIT = Path("/content/evac_round3_auto_kit")

print("KIT =", KIT)
print("KIT 존재:", KIT.exists())
print("수집 스크립트 존재:",
      (KIT / "scripts" / "collect_round3_real.py").exists())
print("모델 존재:",
      (KIT / "seed_model" / "best_round2.pt").exists())

## 1. 공개 라이선스 실제 대피도 후보 60장 수집
기존 Round 1/2 source URL은 제외합니다. 네트워크 상태에 따라 수집 수가 60장보다 적을 수 있습니다.

In [ ]:
# 1번 빠른 버전
from pathlib import Path
import subprocess

RAW = Path("/content/round3_real_raw")

cmd = [
    "python",
    str(KIT / "scripts" / "collect_round3_real.py"),
    "--out", str(RAW),
    "--target", "30",   # 60 → 30
    "--depth", "2",     # 6 → 2
    "--max-side", "1800",
    "--width", "1800",
    "--clean",
    "--exclude-csv", str(KIT / "manifests" / "exclude_prior_sources.csv"),
    "--curated-csv", str(KIT / "manifests" / "curated_fresh_sources.csv"),
]

print("빠른 실제 대피도 수집 시작...")

result = subprocess.run(cmd)

images = sorted((RAW / "images").glob("*.jpg"))

print("수집 완료")
print("이미지 수:", len(images))

if len(images) == 0:
    raise RuntimeError("이미지가 0장입니다. 수집 스크립트 오류가 있습니다.")

print("✅ 1번 완료 → 2번으로 이동")

## 2. 핵심 클래스 후보 우선순위화
Round 2 모델을 낮은 confidence로 돌려 `stair / you_are_here / exit` 가능성이 높은 후보를 앞쪽으로 정렬합니다. **모델 점수는 정답이 아닙니다.**

In [ ]:
RANK=Path('/content/round3_ranked')
!python {KIT/'scripts'/'rank_critical.py'} --model {SEED} --images {RAW/'images'} --metadata {RAW/'metadata.csv'} --out {RANK} --conf 0.08 --imgsz {IMAGE_SIZE} --top 45
files.download(str(RANK/'critical_ranking.csv'))
print('우선 검수 이미지:',len(list((RANK/'top_images').glob('*.jpg'))))

## 3. 우선순위 상위 이미지 자동 밑라벨
다운로드된 ZIP을 반드시 사람이 수정하세요. 특히 **stair 누락, you_are_here 누락, exit 오탐/누락**을 집중 검수합니다.

In [ ]:
PRE=Path('/content/round3_prelabeled')
!python {KIT/'scripts'/'prelabel_safe.py'} --model {SEED} --images {RANK/'top_images'} --out {PRE} --conf 0.10 --imgsz {IMAGE_SIZE} --overwrite
# 출처 메타데이터도 같이 보존
shutil.copy2(RAW/'metadata.csv',PRE/'metadata_all_candidates.csv')
archive=shutil.make_archive('/content/round3_prelabeled','zip',root_dir=PRE)
print(archive)
files.download(archive)

## 4. 사람 검수
`round3_prelabeled.zip`을 PC에서 열어 누락 추가 / 오탐 삭제 / 클래스 수정을 끝냅니다. 이미지 stem과 TXT stem은 동일하게 유지하세요.

## 5. 검수 완료 ZIP 업로드

In [ ]:
up=files.upload()
review_zip=next(Path(k) for k in up if k.lower().endswith('.zip'))
REVIEWED=Path('/content/round3_reviewed'); shutil.rmtree(REVIEWED,ignore_errors=True); REVIEWED.mkdir()
with zipfile.ZipFile(review_zip) as z: z.extractall(REVIEWED)
# 한 단계 중첩된 폴더 자동 보정
if not (REVIEWED/'images').exists():
    dirs=[d for d in REVIEWED.iterdir() if d.is_dir() and (d/'images').exists() and (d/'labels').exists()]
    if len(dirs)==1: REVIEWED=dirs[0]
print('images',len(list((REVIEWED/'images').glob('*'))),'labels',len(list((REVIEWED/'labels').glob('*.txt'))))

## 6. 핵심 클래스 보장 train/val/test 분할
기본값은 val과 test 각각 `exit/stair/you_are_here` 정답 박스 최소 10개입니다. 이 조건을 못 채우면 **학습하지 말고 데이터부터 더 모으는 게 맞습니다.**

In [ ]:
DATA=Path('/content/round3_dataset')
!python {KIT/'scripts'/'split_reviewed.py'} --reviewed {REVIEWED} --out {DATA} --min-critical 10
ROUND3_YAML=DATA/'data_round3.yaml'
print(ROUND3_YAML.read_text())

## 7. Google Drive 연결 및 Stage A — 15 epoch
Drive에 직접 저장하며 `last.pt`가 있으면 자동 resume합니다.

In [ ]:
drive.mount('/content/drive',force_remount=False)
DRIVE_ROOT=Path('/content/drive/MyDrive/evacuation_checkpoints/round3')
DRIVE_ROOT.mkdir(parents=True,exist_ok=True)
RUN_A='evac_round3_stage_a'; DIR_A=DRIVE_ROOT/RUN_A; LAST_A=DIR_A/'weights'/'last.pt'
if LAST_A.exists():
    print('Stage A resume:',LAST_A); stage_a=YOLO(str(LAST_A)).train(resume=True)
else:
    stage_a=YOLO(str(SEED)).train(data=str(ROUND3_YAML),epochs=15,patience=5,imgsz=IMAGE_SIZE,batch=BATCH,optimizer='AdamW',lr0=3e-4,lrf=.08,weight_decay=5e-4,warmup_epochs=1.0,cos_lr=True,degrees=3,translate=.04,scale=.12,perspective=.0008,hsv_h=.006,hsv_s=.12,hsv_v=.14,fliplr=0,flipud=0,mosaic=.10,mixup=0,close_mosaic=5,amp=True,cache=False,workers=2,project=str(DRIVE_ROOT),name=RUN_A,exist_ok=True,seed=20260815)
BEST_A=DIR_A/'weights'/'best.pt'
print('BEST_A',BEST_A,BEST_A.exists())

## 8. Stage B — 5 epoch 정밀 마무리
Stage A 완료 후 실행합니다. 동시에 실행하지 마세요.

In [ ]:
RUN_B='evac_round3_stage_b'; DIR_B=DRIVE_ROOT/RUN_B; LAST_B=DIR_B/'weights'/'last.pt'
if LAST_B.exists():
    print('Stage B resume:',LAST_B); stage_b=YOLO(str(LAST_B)).train(resume=True)
else:
    assert BEST_A.exists(),'Stage A best.pt가 없습니다.'
    stage_b=YOLO(str(BEST_A)).train(data=str(ROUND3_YAML),epochs=5,patience=3,imgsz=IMAGE_SIZE,batch=BATCH,optimizer='AdamW',lr0=1e-4,lrf=.1,weight_decay=5e-4,warmup_epochs=.5,cos_lr=True,degrees=1.5,translate=.02,scale=.06,perspective=.0003,hsv_h=.003,hsv_s=.07,hsv_v=.08,fliplr=0,flipud=0,mosaic=0,mixup=0,amp=True,cache=False,workers=2,project=str(DRIVE_ROOT),name=RUN_B,exist_ok=True,seed=20260815)
BEST_B=DIR_B/'weights'/'best.pt'
print('BEST_B',BEST_B,BEST_B.exists())

## 9. Stage A/B를 validation으로 비교
**test는 여기서 사용하지 않습니다.**

In [ ]:
def val_score(p):
    r=YOLO(str(p)).val(data=str(ROUND3_YAML),split='val',imgsz=IMAGE_SIZE,verbose=False)
    return r, float(r.box.map50)+0.5*float(r.box.mr)
ra,sa=val_score(BEST_A); rb,sb=val_score(BEST_B)
print('A val',float(ra.box.mp),float(ra.box.mr),float(ra.box.map50),float(ra.box.map),'score',sa)
print('B val',float(rb.box.mp),float(rb.box.mr),float(rb.box.map50),float(rb.box.map),'score',sb)
BEST_ROUND3=BEST_B if sb>=sa else BEST_A
print('선택:',BEST_ROUND3)

## 10. 독립 test에서 최종 목표 판정

In [ ]:
TARGET=Path('/content/round3_target_result.json')
!python {KIT/'scripts'/'evaluate_round3.py'} --model {BEST_ROUND3} --data {ROUND3_YAML} --imgsz {IMAGE_SIZE} --out {TARGET}
print(TARGET.read_text())

## 11. 최종 내보내기

In [ ]:
FINAL=YOLO(str(BEST_ROUND3))
onnx=FINAL.export(format='onnx',imgsz=IMAGE_SIZE,opset=12,simplify=True)
OUT=Path('/content/evac_model_round3_final'); shutil.rmtree(OUT,ignore_errors=True); OUT.mkdir()
for src,name in [(BEST_ROUND3,'best.pt'),(Path(onnx),'best.onnx'),(ROUND3_YAML,'data_round3.yaml'),(TARGET,'target_result.json'),(DATA/'split_report.json','split_report.json')]:
    if Path(src).exists(): shutil.copy2(src,OUT/name)
zipout=shutil.make_archive('/content/evac_model_round3_final','zip',root_dir=OUT)
print(zipout); files.download(zipout)